In [4]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

bc = load_breast_cancer()
X, y = bc.data, bc.target
X

array([[1.799e+01, 1.038e+01, 1.228e+02, ..., 2.654e-01, 4.601e-01,
        1.189e-01],
       [2.057e+01, 1.777e+01, 1.329e+02, ..., 1.860e-01, 2.750e-01,
        8.902e-02],
       [1.969e+01, 2.125e+01, 1.300e+02, ..., 2.430e-01, 3.613e-01,
        8.758e-02],
       ...,
       [1.660e+01, 2.808e+01, 1.083e+02, ..., 1.418e-01, 2.218e-01,
        7.820e-02],
       [2.060e+01, 2.933e+01, 1.401e+02, ..., 2.650e-01, 4.087e-01,
        1.240e-01],
       [7.760e+00, 2.454e+01, 4.792e+01, ..., 0.000e+00, 2.871e-01,
        7.039e-02]], shape=(569, 30))

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_zs = scaler.fit_transform(X_train)
X_test_zs = scaler.transform(X_test)

In [3]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense

model = Sequential()
model.add(Input(shape=(30,)))
model.add(Dense(16, activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(1, activation='sigmoid'))
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape          ┃      Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ dense (Dense)                 │ (None, 16)            │          496 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ dense_1 (Dense)               │ (None, 8)             │          136 │
├───────────────────────────────┼───────────────────────┼──────────────┤
│ dense_2 (Dense)               │ (None, 1)             │            9 │
└───────────────────────────────┴───────────────────────┴──────────────┘

 Total params: 641 (2.50 KB)

 Trainable params: 641 (2.50 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

In [7]:
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

history = model.fit(
    X_train_zs, y_train,
    epochs=500,
    validation_split=0.2,
    callbacks=[es],
    verbose=0,
)

In [8]:
from sklearn.metrics import accuracy_score, confusion_matrix

proba = model.predict(X_test_zs).flatten()
pred = (proba >= 0.5).astype(int)

print("accuracy:", accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
accuracy: 0.965034965034965
[[51  2]
 [ 3 87]]


In [10]:
# 의료 문제에서는 악성(malignant)을 놓치는 것(False Negative)이 가장 위험함
# accuracy 외에 클래스별 precision/recall/f1을 함께 확인
from sklearn.metrics import classification_report

print(classification_report(y_test, pred, target_names=['malignant', 'benign']))

              precision    recall  f1-score   support

   malignant       0.94      0.96      0.95        53
      benign       0.98      0.97      0.97        90

    accuracy                           0.97       143
   macro avg       0.96      0.96      0.96       143
weighted avg       0.97      0.97      0.97       143



In [11]:
print("멈춘 epoch:", len(history.history['loss']))
print("최적 epoch:", len(history.history['loss']) - 20)   # patience만큼 뺌


멈춘 epoch: 474
최적 epoch: 454
